# 10장 실습 ② — 시계열 예측

**PyTorch 판**

파형에서 다음 값을 맞힙니다.
**기준선을 먼저 재는 것**부터 시작합니다. (7장 §7.2)

## 10.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 10.1 실험대 — 파형에서 다음 값

**기준선을 먼저 잽니다.** 아무것도 안 하는 모델의 성적을 모르면
모델의 성적을 읽을 수 없습니다.

In [ ]:
# 파형에서 다음 값 맞히기. 주기·진폭·위상이 표본마다 다르다.
x, y = data.sine_series(4000, length=40, seed=42)
s = data.split(x, y, val_ratio=0.15, test_ratio=0.15, seed=42)
print(s.summary())

fig, ax = plt.subplots(figsize=(8.5, 3.0))
for k in range(3):
    ax.plot(x[k, :, 0], lw=1.3, alpha=0.85)
    ax.scatter([40], [y[k]], s=45, zorder=5)
ax.set_xlabel("걸음"); ax.grid(alpha=0.3)
ax.set_title("지난 40걸음을 보고 그다음 한 값을 맞힙니다")
plt.show()

# ★ 기준선을 먼저 잽니다 (7장 §7.2)
baseline = metrics.mae(s.y_test, s.x_test[:, -1, 0])
dlbook.record("ch10_forecast_baseline_mae", baseline)
print(f"기준선 — '마지막 값을 그대로 답한다' MAE {baseline:.4f}")
print("→ 앞으로 나오는 MAE는 이 값과 견주어 읽으십시오.")

## 10.2 학습 함수 — 여기만 판마다 다릅니다

**PyTorch 판에 `Recurrent` 래퍼가 하나 더 있는 것**에 주목하십시오.
PyTorch의 순환 층은 (출력 전체, 마지막 상태)를 돌려주므로,
Keras의 기본 동작(마지막 것만)과 맞추려면 감싸야 합니다.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

device = "cuda" if torch.cuda.is_available() else "cpu"

class Recurrent(nn.Module):
    """마지막 걸음의 상태만 꺼내 쓰는 순환 층.

    PyTorch의 RNN/LSTM/GRU는 (출력 전체, 마지막 상태)를 돌려준다.
    Keras의 기본 동작(마지막 것만)과 맞추려면 이렇게 감싼다.
    """
    def __init__(self, kind, n_in, units, n_out):
        super().__init__()
        C = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[kind]
        self.rnn = C(n_in, units, batch_first=True)
        self.head = nn.Linear(units, n_out)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1])              # 마지막 걸음만

def _train(model, xa, ya, xb, yb, loss_fn, lr, epochs, bs=64, cls=True):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    yt = torch.tensor(ya, dtype=torch.long if cls else torch.float32)
    if not cls:
        yt = yt.view(-1, 1)
    dl = DataLoader(TensorDataset(torch.tensor(xa, dtype=torch.float32), yt),
                    batch_size=bs, shuffle=True)
    for _ in range(dlbook.smoke.epochs(epochs)):
        model.train()
        for xx, yy in dl:
            xx, yy = xx.to(device), yy.to(device)
            opt.zero_grad(); loss_fn(model(xx), yy).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        out = model(torch.tensor(xb, dtype=torch.float32).to(device)).cpu().numpy()
    return model, out

def train_seq(kind, sp, length, lr=0.001, seed=42, epochs=25):
    """분류: 표시된 자리의 부호 맞히기. (시험 정확도)

    이 함수 하나만 판마다 다르다. 아래의 모든 실험 셀은 세 판이 같다.
    """
    dlbook.set_seed(seed)
    if kind == "dnn":
        model = nn.Sequential(nn.Flatten(), nn.Linear(length * 2, 64),
                              nn.ReLU(), nn.Linear(64, 2))
    else:
        model = Recurrent(kind, 2, 32, 2)
    _, out = _train(model, sp.x_train, sp.y_train, sp.x_test, sp.y_test,
                    nn.CrossEntropyLoss(), lr, epochs)
    return metrics.accuracy(sp.y_test, out.argmax(1))

def train_forecast(kind, sp, lr=0.003, seed=42, epochs=30):
    """회귀: 다음 값 맞히기. (MAE, 파라미터 수)"""
    dlbook.set_seed(seed)
    length = sp.x_train.shape[1]
    if kind == "dnn":
        model = nn.Sequential(nn.Flatten(), nn.Linear(length, 64),
                              nn.ReLU(), nn.Linear(64, 1))
    else:
        model = Recurrent(kind, 1, 32, 1)
    model, out = _train(model, sp.x_train, sp.y_train, sp.x_test, sp.y_test,
                        nn.MSELoss(), lr, epochs, cls=False)
    n_params = sum(p.numel() for p in model.parameters())
    return metrics.mae(sp.y_test, out.reshape(-1)), n_params

## 10.3 네 가지 구조를 나란히

In [ ]:
print(f"{'모델':<10}{'파라미터':>12}{'MAE':>10}")
print(f"{'(기준선)':<10}{'-':>12}{baseline:>10.4f}")
for kind in ("dnn", "rnn", "lstm", "gru"):
    mae, n_params = train_forecast(kind, s)
    print(f"{kind:<10}{n_params:>12,}{mae:>10.4f}")
    dlbook.record(f"ch10_forecast_{kind}_mae", mae)

print()
print("→ SimpleRNN이 DNN과 같은 성능을 **파라미터 절반 이하**로 냅니다.")
print("→ LSTM·GRU가 11%쯤 더 낫습니다. 이 과제에서는 게이트가 도움이 됐습니다.")

## 정리

- **기준선을 먼저 재십시오.** 시계열에서 그것은 *"마지막 값을 그대로 답하기"*
  입니다. 이 기준선을 못 넘는 모델이 놀랄 만큼 흔합니다.
- **SimpleRNN이 DNN과 같은 성능을 파라미터 절반 이하로** 냅니다.
  8장에서 CNN이 그랬던 것과 같은 이유 — **파라미터 공유**입니다.
- LSTM·GRU가 11%쯤 낫습니다. 이 과제에서는 게이트가 도움이 됐습니다.

### 연습

1. 예측 구간(`horizon`)을 1 → 5 → 10으로 늘리십시오. 기준선과의 차이가
   어떻게 변합니까.
2. 잡음(`noise`)을 키우면 어떻게 됩니까. 어느 지점에서 기준선을 못 넘습니까.
3. **하나의 긴 시계열**을 잘라 쓰는 경우로 바꿔 보십시오.
   그때는 **반드시 시간 순서로** 나눠야 합니다. (4장 §4.2)